# FPA FOD Comprehensive Analysis: The Full Fire Size Spectrum in California

**Dataset**: Fire Program Analysis — Fire Occurrence Database (FPA FOD), 6th Edition
**Author**: Karen C. Short, USDA Forest Service (2022)
**DOI**: [10.2737/RDS-2013-0009.6](https://doi.org/10.2737/RDS-2013-0009.6)
**Period**: 1992–2020 | **National Records**: 2.3 million | **California Records**: 251,881

---

### Why FPA FOD Matters

Previous analyses used **CAL FIRE fire perimeters** (polygons, all sizes but variable quality) and **MTBS** (satellite-based severity, >= 1,000 acres). FPA FOD fills a critical gap:

| Feature | CAL FIRE | MTBS | **FPA FOD** |
|---------|----------|------|-------------|
| Geometry | Polygon perimeters | Point centroids | **Point locations** |
| Size threshold | None (all sizes) | >= 1,000 ac (west) | **All sizes (0.01+ ac)** |
| Size spectrum | Biased toward large | Large fires only | **93% under 10 acres** |
| Cause codes | 19 numeric codes | Incident type only | **13 NWCG standardized** |
| Severity data | None | dNBR thresholds | None |
| Land ownership | Agency only | None | **OWNER_DESCR field** |
| County/FIPS | None | None | **County + FIPS code** |
| Fire duration | Alarm → Containment | None | **Discovery → Containment** |
| Time of day | None | None | **Discovery time (HHMM)** |
| MTBS linkage | None | Event_ID | **MTBS_ID field** |
| Year range | 1878–2025 | 1984–2024 | **1992–2020** |

## Learning Objectives

1. **Full Size Spectrum** — Understand why 93% of fires are invisible to MTBS/perimeter datasets
2. **NWCG Cause Taxonomy** — Learn the standardized 13-category cause classification
3. **Temporal & Seasonal Patterns** — Discover time-of-day and duration patterns by cause
4. **Land Ownership** — Analyze fire patterns across federal, state, and private lands
5. **County Geography** — Explore sub-state spatial patterns with FIPS codes
6. **Triple Cross-Reference** — Link FPA FOD ↔ MTBS (via MTBS_ID) ↔ CAL FIRE (via year)
7. **ML Feature Engineering** — Identify new features for Phase 2–4 model development

---

## Table of Contents

1. [Introduction & Context](#part1) (above)
2. [Data Loading & Quality](#part2)
3. [National Context](#part3)
4. [Temporal Analysis](#part4)
5. [Seasonal Patterns & Duration](#part5)
6. [Fire Size Spectrum](#part6)
7. [Cause Attribution](#part7)
8. [Spatial Analysis](#part8)
9. [Cross-Reference & ML Readiness](#part9)

<a id='part2'></a>

---

# Part 2: Data Loading & Quality Assessment

In [ ]:
# Standard library imports
import sys
import warnings
from pathlib import Path

import gc
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Project imports
sys.path.insert(0, str(Path.cwd().parent))
from src.utils.constants import (
    FPA_FOD_PATH, CRS_ALBERS, CRS_WGS84, CRS_NAD83,
    FPA_START_YEAR, FPA_END_YEAR, FIGURES_DIR, FIGURE_DPI,
    MEGA_FIRE_THRESHOLD, LARGE_FIRE_THRESHOLD, OUTPUT_DIR,
)
from src.utils.fpa_data_loader import (
    load_fpa_fod, load_fpa_national_summary, add_fpa_derived_columns,
    reproject_to_albers, get_fpa_data_quality_summary,
)
from src.utils.fpa_domain_mappings import (
    NWCG_CAUSE_CLASSIFICATION_MAP, NWCG_GENERAL_CAUSE_MAP,
    NWCG_REPORTING_AGENCY_MAP, FIRE_SIZE_CLASS_MAP, FIRE_SIZE_CLASS_ORDER,
    OWNER_DESCR_STANDARDIZED, SOURCE_SYSTEM_TYPE_MAP,
    get_nwcg_cause_label, get_general_cause_label,
    get_fire_size_class_label, get_owner_standardized,
)
from src.utils.plotting import (
    setup_plotting_style, save_figure, format_acres, get_month_labels,
    add_data_period_annotation,
    get_nwcg_cause_colors, get_general_cause_colors,
    get_fire_size_class_colors, get_source_system_colors, get_owner_colors,
)

# Setup
setup_plotting_style()
FPA_FIGURES_DIR = FIGURES_DIR / "fpa"
FPA_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"FPA FOD GeoPackage: {FPA_FOD_PATH}")
print(f"File exists: {FPA_FOD_PATH.exists()}")
print(f"Figures output: {FPA_FIGURES_DIR}")

In [ ]:
# Load California FPA FOD data (SQL WHERE for efficiency)
print("Loading California FPA FOD data...")
ca = load_fpa_fod(state='CA')
print(f"Loaded {len(ca):,} California fire records")

# Add derived columns
ca = add_fpa_derived_columns(ca)
print(f"Year range: {ca['FIRE_YEAR'].min()} - {ca['FIRE_YEAR'].max()}")
print(f"Total acres burned: {ca['FIRE_SIZE'].sum():,.0f}")
print(f"CRS: {ca.crs}")

In [ ]:
# Data quality summary
quality = get_fpa_data_quality_summary(ca)
print("=" * 60)
print("FPA FOD California — Data Quality Summary")
print("=" * 60)
for key, val in quality.items():
    print(f"  {key:35s}: {val}")

In [ ]:
# Figure 1: Data Quality & Completeness
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Field completeness
fields = ['FIRE_YEAR', 'DISCOVERY_DATE', 'FIRE_SIZE', 'NWCG_CAUSE_CLASSIFICATION',
          'NWCG_GENERAL_CAUSE', 'NWCG_REPORTING_AGENCY', 'OWNER_DESCR',
          'STATE', 'COUNTY', 'FIPS_CODE', 'CONT_DATE', 'DISCOVERY_TIME',
          'MTBS_ID', 'FIRE_NAME']
pcts = [(ca[f].notna().sum() / len(ca)) * 100 for f in fields]
labels = [f.replace('NWCG_', '').replace('_', ' ').title() for f in fields]

colors = ['#2ECC71' if p >= 95 else '#F39C12' if p >= 50 else '#E74C3C' for p in pcts]
bars = axes[0].barh(range(len(fields)), pcts, color=colors)
axes[0].set_yticks(range(len(fields)))
axes[0].set_yticklabels(labels, fontsize=9)
axes[0].set_xlabel('Completeness (%)')
axes[0].set_title('Field Completeness — California FPA FOD')
axes[0].set_xlim(0, 105)
axes[0].axvline(x=95, color='green', linestyle='--', alpha=0.5, label='95%')
axes[0].axvline(x=50, color='orange', linestyle='--', alpha=0.5, label='50%')
axes[0].legend(fontsize=8)
for bar, pct in zip(bars, pcts):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{pct:.1f}%', va='center', fontsize=8)

# Right: Completeness by decade for key optional fields
optional_fields = ['CONT_DATE', 'DISCOVERY_TIME', 'COUNTY', 'MTBS_ID', 'FIRE_NAME']
decades = sorted(ca['Decade'].dropna().unique())
decade_data = {}
for field in optional_fields:
    decade_data[field] = []
    for dec in decades:
        mask = ca['Decade'] == dec
        pct = (ca.loc[mask, field].notna().sum() / mask.sum()) * 100
        decade_data[field].append(pct)

x = np.arange(len(decades))
width = 0.15
for i, field in enumerate(optional_fields):
    label = field.replace('NWCG_', '').replace('_', ' ').title()
    axes[1].bar(x + i * width, decade_data[field], width, label=label, alpha=0.85)

axes[1].set_xticks(x + width * 2)
axes[1].set_xticklabels([f"{int(d)}s" for d in decades])
axes[1].set_ylabel('Completeness (%)')
axes[1].set_title('Optional Field Completeness by Decade')
axes[1].legend(fontsize=8, loc='upper left')
axes[1].set_ylim(0, 110)

fig.suptitle('Figure 1: FPA FOD Data Quality & Completeness', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/01_data_quality_completeness')
plt.show()

<a id='part3'></a>

---

# Part 3: National Context — California Among All States

In [ ]:
# Load national summary (minimal columns for memory efficiency)
print("Loading national summary (minimal columns)...")
national = load_fpa_national_summary()
print(f"National records: {len(national):,}")
print(f"States: {national['STATE'].nunique()}")

In [ ]:
# Figure 2: State Comparison — Top 15 by count, acres, median size
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# By count
state_counts = national.groupby('STATE').size().sort_values(ascending=False).head(15)
colors_count = ['#E63946' if s == 'CA' else '#457B9D' for s in state_counts.index]
state_counts.plot(kind='barh', ax=axes[0], color=colors_count)
axes[0].set_title('Fire Count')
axes[0].set_xlabel('Number of Fires')
for i, (val, state) in enumerate(zip(state_counts.values, state_counts.index)):
    axes[0].text(val + state_counts.max() * 0.01, i, f'{val:,.0f}', va='center', fontsize=8)

# By total acres
state_acres = national.groupby('STATE')['FIRE_SIZE'].sum().sort_values(ascending=False).head(15)
colors_acres = ['#E63946' if s == 'CA' else '#F4A261' for s in state_acres.index]
state_acres.plot(kind='barh', ax=axes[1], color=colors_acres)
axes[1].set_title('Total Acres Burned')
axes[1].set_xlabel('Acres')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))

# By median size
state_median = national.groupby('STATE')['FIRE_SIZE'].median().sort_values(ascending=False).head(15)
colors_median = ['#E63946' if s == 'CA' else '#2A9D8F' for s in state_median.index]
state_median.plot(kind='barh', ax=axes[2], color=colors_median)
axes[2].set_title('Median Fire Size (acres)')
axes[2].set_xlabel('Acres')

fig.suptitle('Figure 2: FPA FOD State Comparison — Top 15 (CA highlighted)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/02_state_comparison')
plt.show()

# Cleanup national data
del national
gc.collect()
print("National data freed from memory")

In [ ]:
# Figure 3: California Fire Size Class Distribution vs National
# Reload minimal national data for size class comparison
print("Loading national size class summary...")
national_sc = load_fpa_national_summary(columns=['STATE', 'FIRE_SIZE_CLASS'])

# National size class distribution
nat_sc = national_sc['FIRE_SIZE_CLASS'].value_counts(normalize=True).reindex(FIRE_SIZE_CLASS_ORDER)
ca_sc = ca['FIRE_SIZE_CLASS'].value_counts(normalize=True).reindex(FIRE_SIZE_CLASS_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc_colors = get_fire_size_class_colors()

# Left: Side-by-side comparison
x = np.arange(len(FIRE_SIZE_CLASS_ORDER))
width = 0.35
bars1 = axes[0].bar(x - width/2, nat_sc.values * 100, width, label='National',
                     color='#457B9D', alpha=0.8)
bars2 = axes[0].bar(x + width/2, ca_sc.values * 100, width, label='California',
                     color='#E63946', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([get_fire_size_class_label(c) for c in FIRE_SIZE_CLASS_ORDER], rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Percentage of Fires (%)')
axes[0].set_title('Fire Size Class Distribution')
axes[0].legend()

# Right: CA size class by count with colors
ca_sc_counts = ca['FIRE_SIZE_CLASS'].value_counts().reindex(FIRE_SIZE_CLASS_ORDER)
bar_colors = [sc_colors[c] for c in FIRE_SIZE_CLASS_ORDER]
bars = axes[1].bar(range(len(FIRE_SIZE_CLASS_ORDER)), ca_sc_counts.values, color=bar_colors)
axes[1].set_xticks(range(len(FIRE_SIZE_CLASS_ORDER)))
axes[1].set_xticklabels([get_fire_size_class_label(c) for c in FIRE_SIZE_CLASS_ORDER], rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Number of Fires')
axes[1].set_title('California Fire Count by Size Class')
axes[1].set_yscale('log')
for bar, val in zip(bars, ca_sc_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                 f'{val:,.0f}', ha='center', fontsize=8)

fig.suptitle('Figure 3: Fire Size Class — National vs California', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/03_size_class_comparison')
plt.show()

del national_sc
gc.collect()

<a id='part4'></a>

---

# Part 4: Temporal Analysis — 29 Years of California Fires (1992–2020)

In [ ]:
# Figure 4: Annual timeline by cause classification
yearly = ca.groupby(['FIRE_YEAR', 'Cause_Label']).agg(
    count=('FIRE_SIZE', 'size'),
    total_acres=('FIRE_SIZE', 'sum')
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
cause_colors = get_nwcg_cause_colors()

# Top: Fire count by cause
for cause in ['Human', 'Natural', 'Missing']:
    mask = yearly['Cause_Label'] == cause
    axes[0].plot(yearly.loc[mask, 'FIRE_YEAR'], yearly.loc[mask, 'count'],
                 color=cause_colors[cause], label=cause, linewidth=2, marker='o', markersize=3)
axes[0].set_ylabel('Number of Fires')
axes[0].set_title('Annual Fire Count by Cause Classification')
axes[0].legend()
add_data_period_annotation(axes[0], FPA_START_YEAR, FPA_END_YEAR, len(ca))

# Bottom: Total acres by year
yearly_total = ca.groupby('FIRE_YEAR').agg(
    total_acres=('FIRE_SIZE', 'sum'),
    count=('FIRE_SIZE', 'size')
).reset_index()

axes[1].bar(yearly_total['FIRE_YEAR'], yearly_total['total_acres'],
            color='#E63946', alpha=0.7, label='Total Acres')
axes[1].set_ylabel('Total Acres Burned')
axes[1].set_xlabel('Year')
axes[1].set_title('Annual Total Acres Burned')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))

# Add trend line
years = yearly_total['FIRE_YEAR'].values
acres = yearly_total['total_acres'].values
slope, intercept, r, p, se = stats.linregress(years, acres)
trend_line = slope * years + intercept
axes[1].plot(years, trend_line, 'k--', linewidth=2, alpha=0.6,
             label=f'Trend (R²={r**2:.3f}, p={p:.4f})')
axes[1].legend()

# Highlight worst year
worst_idx = yearly_total['total_acres'].idxmax()
worst_year = yearly_total.loc[worst_idx, 'FIRE_YEAR']
worst_acres = yearly_total.loc[worst_idx, 'total_acres']
axes[1].annotate(f'{worst_year}\n{format_acres(worst_acres)} acres',
                 xy=(worst_year, worst_acres),
                 xytext=(worst_year - 5, worst_acres * 0.85),
                 arrowprops=dict(arrowstyle='->', color='black'),
                 fontsize=10, fontweight='bold')

fig.suptitle('Figure 4: FPA FOD California — 29-Year Timeline (1992–2020)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/04_temporal_timeline')
plt.show()

In [ ]:
# Figure 5: Decade Comparison (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
decades = sorted(ca['Decade'].dropna().unique())
decade_labels = [f"{int(d)}s" for d in decades]

# Panel 1: Fire count by decade
dec_count = ca.groupby('Decade').size()
axes[0,0].bar(decade_labels, dec_count.values, color='#3498DB', alpha=0.8)
axes[0,0].set_title('Total Fire Count by Decade')
axes[0,0].set_ylabel('Number of Fires')
for i, val in enumerate(dec_count.values):
    axes[0,0].text(i, val + dec_count.max() * 0.02, f'{val:,.0f}', ha='center', fontsize=9)

# Panel 2: Total acres by decade
dec_acres = ca.groupby('Decade')['FIRE_SIZE'].sum()
axes[0,1].bar(decade_labels, dec_acres.values, color='#E63946', alpha=0.8)
axes[0,1].set_title('Total Acres Burned by Decade')
axes[0,1].set_ylabel('Acres')
axes[0,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))
for i, val in enumerate(dec_acres.values):
    axes[0,1].text(i, val + dec_acres.max() * 0.02, format_acres(val), ha='center', fontsize=9)

# Panel 3: Mean fire size by decade
dec_mean = ca.groupby('Decade')['FIRE_SIZE'].mean()
axes[1,0].bar(decade_labels, dec_mean.values, color='#F39C12', alpha=0.8)
axes[1,0].set_title('Mean Fire Size by Decade (acres)')
axes[1,0].set_ylabel('Mean Acres')
for i, val in enumerate(dec_mean.values):
    axes[1,0].text(i, val + dec_mean.max() * 0.02, f'{val:.1f}', ha='center', fontsize=9)

# Panel 4: Mega-fire count by decade
dec_mega = ca[ca['FIRE_SIZE'] >= MEGA_FIRE_THRESHOLD].groupby('Decade').size().reindex(decades, fill_value=0)
axes[1,1].bar(decade_labels, dec_mega.values, color='#922B21', alpha=0.8)
axes[1,1].set_title(f'Mega-Fire Count (>= {MEGA_FIRE_THRESHOLD:,} ac) by Decade')
axes[1,1].set_ylabel('Count')
for i, val in enumerate(dec_mega.values):
    axes[1,1].text(i, val + 0.2, f'{val}', ha='center', fontsize=10, fontweight='bold')

fig.suptitle('Figure 5: FPA FOD California — Decade Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/05_decade_comparison')
plt.show()

<a id='part5'></a>

---

# Part 5: Seasonal Patterns & Duration Analysis

FPA FOD uniquely provides **discovery time-of-day** and **containment date** — enabling
duration analysis and diurnal pattern discovery unavailable in CAL FIRE or MTBS.

In [ ]:
# Figure 6: Monthly Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
month_labels = get_month_labels()

# Left: Fire count by month
monthly_count = ca.groupby('Discovery_Month').size()
season_colors_monthly = []
for m in range(1, 13):
    if m in (6, 7, 8, 9):
        season_colors_monthly.append('#E63946')  # High Risk
    elif m in (10, 11, 12, 1):
        season_colors_monthly.append('#F39C12')  # Transition
    else:
        season_colors_monthly.append('#3498DB')  # Low Risk

axes[0].bar(range(1, 13), monthly_count.reindex(range(1, 13), fill_value=0).values,
            color=season_colors_monthly)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_labels)
axes[0].set_ylabel('Number of Fires')
axes[0].set_title('Monthly Fire Count')

# Add season legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#E63946', label='High Risk (Jun–Sep)'),
                   Patch(facecolor='#F39C12', label='Transition (Oct–Jan)'),
                   Patch(facecolor='#3498DB', label='Low Risk (Feb–May)')]
axes[0].legend(handles=legend_elements, fontsize=8)

# Right: Total acres by month
monthly_acres = ca.groupby('Discovery_Month')['FIRE_SIZE'].sum()
axes[1].bar(range(1, 13), monthly_acres.reindex(range(1, 13), fill_value=0).values,
            color=season_colors_monthly)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_labels)
axes[1].set_ylabel('Total Acres Burned')
axes[1].set_title('Monthly Acres Burned')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))

# Calculate and annotate high-risk season percentage
high_risk_acres = ca[ca['Fire_Season'] == 'High Risk']['FIRE_SIZE'].sum()
total_acres = ca['FIRE_SIZE'].sum()
pct = 100 * high_risk_acres / total_acres
axes[1].annotate(f'Jun–Sep: {pct:.0f}% of burned area',
                 xy=(7.5, monthly_acres.max() * 0.9),
                 fontsize=11, fontweight='bold', ha='center',
                 bbox=dict(boxstyle='round', facecolor='#E63946', alpha=0.2))

fig.suptitle('Figure 6: FPA FOD California — Monthly Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/06_monthly_distribution')
plt.show()

In [ ]:
# Figure 7: Fire Duration Analysis (unique to FPA FOD)
has_duration = ca[ca['Duration_Days'].notna() & (ca['Duration_Days'] >= 0)].copy()
print(f"Records with valid duration: {len(has_duration):,} ({100*len(has_duration)/len(ca):.1f}%)")
print(f"Median duration: {has_duration['Duration_Days'].median():.1f} days")
print(f"Mean duration: {has_duration['Duration_Days'].mean():.1f} days")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Left: Duration distribution (log scale)
durations = has_duration['Duration_Days'].clip(upper=365)
axes[0].hist(durations, bins=100, color='#3498DB', alpha=0.7, edgecolor='white')
axes[0].set_xlabel('Duration (days)')
axes[0].set_ylabel('Count')
axes[0].set_title('Fire Duration Distribution')
axes[0].axvline(x=has_duration['Duration_Days'].median(), color='red', linestyle='--',
                label=f"Median: {has_duration['Duration_Days'].median():.1f} days")
axes[0].legend()
axes[0].set_yscale('log')

# Middle: Duration by size class
size_order = FIRE_SIZE_CLASS_ORDER
dur_by_size = has_duration[has_duration['FIRE_SIZE_CLASS'].isin(size_order)]
sc_colors = get_fire_size_class_colors()
boxprops = dict(alpha=0.8)
bp = dur_by_size.boxplot(column='Duration_Days', by='FIRE_SIZE_CLASS',
                          ax=axes[1], positions=range(len(size_order)),
                          showfliers=False, patch_artist=True,
                          return_type='dict')
# Color the boxes
for patch, cls in zip(bp['Duration_Days']['boxes'], size_order):
    patch.set_facecolor(sc_colors.get(cls, '#95A5A6'))
axes[1].set_xticklabels([get_fire_size_class_label(c) for c in size_order], rotation=45, ha='right', fontsize=8)
axes[1].set_xlabel('Fire Size Class')
axes[1].set_ylabel('Duration (days)')
axes[1].set_title('Duration by Size Class')
axes[1].set_ylim(0, 100)
fig.suptitle('')  # Remove auto-generated boxplot title
axes[1].set_title('Duration by Size Class')

# Right: Duration by cause
cause_order = ['Lightning', 'Arson', 'Debris Burning', 'Equipment Use',
               'Powerline', 'Smoking', 'Missing/Unknown']
dur_by_cause = has_duration[has_duration['General_Cause_Label'].isin(cause_order)]
gc_colors = get_general_cause_colors()
bp2 = dur_by_cause.boxplot(column='Duration_Days', by='General_Cause_Label',
                            ax=axes[2], positions=range(len(cause_order)),
                            showfliers=False, patch_artist=True,
                            return_type='dict')
for patch, cause in zip(bp2['Duration_Days']['boxes'], cause_order):
    patch.set_facecolor(gc_colors.get(cause, '#95A5A6'))
axes[2].set_xticklabels(cause_order, rotation=45, ha='right', fontsize=8)
axes[2].set_xlabel('Cause')
axes[2].set_ylabel('Duration (days)')
axes[2].set_title('Duration by Cause')
axes[2].set_ylim(0, 30)
fig.suptitle('')
plt.suptitle('Figure 7: Fire Duration — Discovery to Containment (FPA FOD unique)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/07_fire_duration')
plt.show()

In [ ]:
# Figure 8: Discovery Time-of-Day (FPA FOD unique)
has_time = ca[ca['Discovery_Hour'].notna()].copy()
print(f"Records with discovery time: {len(has_time):,} ({100*len(has_time)/len(ca):.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Overall hourly distribution
hourly = has_time.groupby('Discovery_Hour').size()
axes[0].bar(hourly.index, hourly.values, color='#F39C12', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Fires')
axes[0].set_title('Discovery Time Distribution (All Fires)')
axes[0].set_xticks(range(0, 24, 2))
axes[0].set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], rotation=45)
peak_hour = hourly.idxmax()
axes[0].annotate(f'Peak: {int(peak_hour):02d}:00',
                 xy=(peak_hour, hourly.max()),
                 xytext=(peak_hour + 3, hourly.max() * 0.95),
                 arrowprops=dict(arrowstyle='->'), fontsize=10, fontweight='bold')

# Right: Hourly pattern by cause (Human vs Natural)
cause_colors = get_nwcg_cause_colors()
for cause in ['Human', 'Natural']:
    mask = has_time['Cause_Label'] == cause
    hourly_cause = has_time[mask].groupby('Discovery_Hour').size()
    # Normalize to percentage
    hourly_pct = hourly_cause / hourly_cause.sum() * 100
    axes[1].plot(hourly_pct.index, hourly_pct.values, color=cause_colors[cause],
                 label=cause, linewidth=2.5, marker='o', markersize=4)

axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Percentage of Fires (%)')
axes[1].set_title('Diurnal Pattern: Human vs Natural Fires')
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], rotation=45)
axes[1].legend(fontsize=11)
axes[1].annotate('Human fires peak\nafternoon (13–15h)',
                 xy=(14, axes[1].get_ylim()[1] * 0.85),
                 fontsize=9, ha='center',
                 bbox=dict(boxstyle='round', facecolor='#E74C3C', alpha=0.15))
axes[1].annotate('Lightning fires peak\nlate afternoon/evening',
                 xy=(17, axes[1].get_ylim()[1] * 0.55),
                 fontsize=9, ha='center',
                 bbox=dict(boxstyle='round', facecolor='#2ECC71', alpha=0.15))

fig.suptitle('Figure 8: Discovery Time-of-Day (FPA FOD unique)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/08_discovery_time_of_day')
plt.show()

<a id='part6'></a>

---

# Part 6: Fire Size Spectrum — Small Fire Dominance

FPA FOD's defining feature: **93% of California fires are under 10 acres** — a vast population
invisible to MTBS (>= 1,000 ac) and underrepresented in CAL FIRE perimeters.

In [ ]:
# Figure 9: Full Fire Size Spectrum
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Raw histogram
axes[0,0].hist(ca['FIRE_SIZE'], bins=200, color='#E63946', alpha=0.7, edgecolor='white')
axes[0,0].set_xlabel('Fire Size (acres)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Raw Distribution (truncated at 1K ac)')
axes[0,0].set_xlim(0, 1000)

# Panel 2: Log-scale histogram
axes[0,1].hist(ca['Log10_FireSize'], bins=100, color='#3498DB', alpha=0.7, edgecolor='white')
axes[0,1].set_xlabel('log₁₀(Fire Size in acres)')
axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Log-Transformed Distribution')
axes[0,1].axvline(x=np.log10(10), color='red', linestyle='--', alpha=0.7, label='10 acres')
axes[0,1].axvline(x=np.log10(1000), color='orange', linestyle='--', alpha=0.7, label='1,000 acres')
axes[0,1].legend()

# Panel 3: Pareto / Lorenz curve
sorted_sizes = ca['FIRE_SIZE'].sort_values().values
cumsum = np.cumsum(sorted_sizes)
cumsum_pct = cumsum / cumsum[-1] * 100
fire_pct = np.arange(1, len(sorted_sizes) + 1) / len(sorted_sizes) * 100

axes[1,0].plot(fire_pct, cumsum_pct, color='#E63946', linewidth=2)
axes[1,0].plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Perfect equality')
axes[1,0].set_xlabel('Cumulative % of Fires (smallest → largest)')
axes[1,0].set_ylabel('Cumulative % of Burned Area')
axes[1,0].set_title('Pareto / Lorenz Curve')

# Annotate key thresholds
for pct_fire, label in [(99, 'Top 1%'), (90, 'Top 10%')]:
    idx = int(len(sorted_sizes) * pct_fire / 100)
    pct_area = 100 - cumsum_pct[idx]
    axes[1,0].annotate(f'{label}: {pct_area:.0f}% of area',
                       xy=(pct_fire, cumsum_pct[idx]),
                       xytext=(pct_fire - 25, cumsum_pct[idx] - 10),
                       arrowprops=dict(arrowstyle='->'),
                       fontsize=9, fontweight='bold')
axes[1,0].legend()

# Panel 4: Size category breakdown
size_cats = ['Tiny (<10)', 'Minor (10-100)', 'Small (100-1K)',
             'Medium (1K-10K)', 'Large (10K-100K)', 'Mega (100K+)']
cat_counts = ca['Size_Category'].value_counts().reindex(size_cats, fill_value=0)
cat_acres = ca.groupby('Size_Category')['FIRE_SIZE'].sum().reindex(size_cats, fill_value=0)

x = np.arange(len(size_cats))
width = 0.35
ax2 = axes[1,1].twinx()
bars1 = axes[1,1].bar(x - width/2, cat_counts.values, width, color='#3498DB', alpha=0.8, label='Count')
bars2 = ax2.bar(x + width/2, cat_acres.values, width, color='#E63946', alpha=0.8, label='Acres')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(size_cats, rotation=45, ha='right', fontsize=8)
axes[1,1].set_ylabel('Number of Fires', color='#3498DB')
ax2.set_ylabel('Total Acres', color='#E63946')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))
axes[1,1].set_title('Count vs Acres by Size Category')
axes[1,1].set_yscale('log')

# Combined legend
lines1, labels1 = axes[1,1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1,1].legend(lines1 + lines2, labels1 + labels2, loc='upper left')

fig.suptitle('Figure 9: Full Fire Size Spectrum — The Long Tail',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/09_fire_size_spectrum')
plt.show()

In [ ]:
# Figure 10: Small Fire Dominance — The Invisible Majority
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Pie chart of fire count by size category
size_cats = ['Tiny (<10)', 'Minor (10-100)', 'Small (100-1K)',
             'Medium (1K-10K)', 'Large (10K-100K)', 'Mega (100K+)']
cat_counts = ca['Size_Category'].value_counts().reindex(size_cats, fill_value=0)
pie_colors = ['#2ECC71', '#82E0AA', '#F9E79F', '#F5B041', '#E74C3C', '#922B21']
explode = (0.05, 0, 0, 0, 0, 0)

axes[0].pie(cat_counts.values, labels=size_cats, autopct='%1.1f%%',
            colors=pie_colors, explode=explode, startangle=90, pctdistance=0.85,
            textprops={'fontsize': 9})
axes[0].set_title('Fire COUNT by Size Category\n(93% of fires < 10 acres)')

# Right: Pie chart of burned area
cat_acres = ca.groupby('Size_Category')['FIRE_SIZE'].sum().reindex(size_cats, fill_value=0)
explode2 = (0, 0, 0, 0, 0, 0.05)
axes[1].pie(cat_acres.values, labels=size_cats, autopct='%1.1f%%',
            colors=pie_colors, explode=explode2, startangle=90, pctdistance=0.85,
            textprops={'fontsize': 9})
axes[1].set_title('Burned AREA by Size Category\n(mega-fires dominate acreage)')

# Summary stats
under_10 = (ca['FIRE_SIZE'] < 10).sum()
over_1k = (ca['FIRE_SIZE'] >= 1000).sum()
over_1k_acres = ca[ca['FIRE_SIZE'] >= 1000]['FIRE_SIZE'].sum()
total_acres = ca['FIRE_SIZE'].sum()
print(f"Fires < 10 acres: {under_10:,} ({100*under_10/len(ca):.1f}%)")
print(f"Fires >= 1,000 acres: {over_1k:,} ({100*over_1k/len(ca):.1f}%)")
print(f"Acres from >= 1,000 ac fires: {over_1k_acres:,.0f} ({100*over_1k_acres/total_acres:.1f}%)")

fig.suptitle('Figure 10: Small Fire Dominance — Count vs Area',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/10_small_fire_dominance')
plt.show()

<a id='part7'></a>

---

# Part 7: Cause Attribution — NWCG Standardized Taxonomy

FPA FOD uses the **NWCG (National Wildfire Coordinating Group)** cause classification,
providing two levels: a broad 3-category classification (Human/Natural/Missing) and a
detailed 13-category general cause taxonomy — far more standardized than CAL FIRE's
19 numeric codes.

In [ ]:
# Figure 11: Cause Classification Overview (Human / Natural / Missing)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cause_colors = get_nwcg_cause_colors()

# Left: Count
cause_counts = ca['Cause_Label'].value_counts()
cause_order = ['Human', 'Natural', 'Missing']
counts = cause_counts.reindex(cause_order, fill_value=0)
bars = axes[0].bar(cause_order, counts.values,
                   color=[cause_colors[c] for c in cause_order])
axes[0].set_ylabel('Number of Fires')
axes[0].set_title('Fire Count by Cause')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max() * 0.02,
                 f'{val:,.0f}\n({100*val/len(ca):.1f}%)', ha='center', fontsize=10)

# Middle: Acres
cause_acres = ca.groupby('Cause_Label')['FIRE_SIZE'].sum().reindex(cause_order, fill_value=0)
bars = axes[1].bar(cause_order, cause_acres.values,
                   color=[cause_colors[c] for c in cause_order])
axes[1].set_ylabel('Total Acres Burned')
axes[1].set_title('Burned Area by Cause')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))
for bar, val in zip(bars, cause_acres.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + cause_acres.max() * 0.02,
                 f'{format_acres(val)}\n({100*val/total_acres:.1f}%)', ha='center', fontsize=10)

# Right: Mean size
cause_mean = ca.groupby('Cause_Label')['FIRE_SIZE'].mean().reindex(cause_order, fill_value=0)
bars = axes[2].bar(cause_order, cause_mean.values,
                   color=[cause_colors[c] for c in cause_order])
axes[2].set_ylabel('Mean Fire Size (acres)')
axes[2].set_title('Mean Fire Size by Cause')
for bar, val in zip(bars, cause_mean.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + cause_mean.max() * 0.02,
                 f'{val:.1f} ac', ha='center', fontsize=10)

fig.suptitle('Figure 11: NWCG Cause Classification Overview',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/11_cause_classification')
plt.show()

In [ ]:
# Figure 12: NWCG General Cause Deep Dive (13 categories)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
gc_colors = get_general_cause_colors()

# Sort by count
gc_counts = ca['General_Cause_Label'].value_counts()
gc_order = gc_counts.index.tolist()

# Left: Count
colors_list = [gc_colors.get(c, '#95A5A6') for c in gc_order]
gc_counts.plot(kind='barh', ax=axes[0], color=colors_list)
axes[0].set_xlabel('Number of Fires')
axes[0].set_title('Fire Count by General Cause')
for i, val in enumerate(gc_counts.values):
    axes[0].text(val + gc_counts.max() * 0.01, i,
                 f'{val:,.0f} ({100*val/len(ca):.1f}%)', va='center', fontsize=8)

# Right: Acres
gc_acres = ca.groupby('General_Cause_Label')['FIRE_SIZE'].sum().reindex(gc_order)
gc_acres.plot(kind='barh', ax=axes[1], color=colors_list)
axes[1].set_xlabel('Total Acres Burned')
axes[1].set_title('Burned Area by General Cause')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))
for i, val in enumerate(gc_acres.values):
    axes[1].text(val + gc_acres.max() * 0.01, i,
                 f'{format_acres(val)} ({100*val/total_acres:.1f}%)', va='center', fontsize=8)

fig.suptitle('Figure 12: NWCG General Cause Taxonomy — 13 Standardized Categories',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/12_general_cause_deep_dive')
plt.show()

In [ ]:
# Figure 13: Cause Trends Over Time
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Cause classification trend
yearly_cause = ca.groupby(['FIRE_YEAR', 'Cause_Label']).size().unstack(fill_value=0)
cause_colors = get_nwcg_cause_colors()
for cause in ['Human', 'Natural', 'Missing']:
    if cause in yearly_cause.columns:
        axes[0].plot(yearly_cause.index, yearly_cause[cause],
                     color=cause_colors[cause], label=cause, linewidth=2)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Fires')
axes[0].set_title('Cause Classification Trend')
axes[0].legend()

# Right: Top 5 general causes over time
top_causes = ca['General_Cause_Label'].value_counts().head(5).index.tolist()
yearly_gc = ca.groupby(['FIRE_YEAR', 'General_Cause_Label']).size().unstack(fill_value=0)
gc_colors = get_general_cause_colors()
for cause in top_causes:
    if cause in yearly_gc.columns:
        axes[1].plot(yearly_gc.index, yearly_gc[cause],
                     color=gc_colors.get(cause, '#95A5A6'), label=cause, linewidth=2)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Fires')
axes[1].set_title('Top 5 General Causes Over Time')
axes[1].legend(fontsize=9)

fig.suptitle('Figure 13: Cause Attribution Trends (1992–2020)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/13_cause_trends')
plt.show()

<a id='part8'></a>

---

# Part 8: Spatial Analysis — Land Ownership, County Geography, and Mapping

FPA FOD provides **land ownership** (OWNER_DESCR) and **county/FIPS** data —
enabling sub-state spatial analysis unavailable in previous datasets.

In [ ]:
# Figure 14: Land Ownership Analysis (FPA FOD unique)
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
owner_colors = get_owner_colors()

# Standardize ownership
owner_counts = ca['Owner_Std'].value_counts()
owner_order = owner_counts.index.tolist()

# Left: Count
colors_list = [owner_colors.get(o, '#95A5A6') for o in owner_order]
owner_counts.plot(kind='barh', ax=axes[0], color=colors_list)
axes[0].set_xlabel('Number of Fires')
axes[0].set_title('Fire Count by Land Owner')
for i, val in enumerate(owner_counts.values):
    axes[0].text(val + owner_counts.max() * 0.01, i,
                 f'{val:,.0f}', va='center', fontsize=8)

# Middle: Acres
owner_acres = ca.groupby('Owner_Std')['FIRE_SIZE'].sum().reindex(owner_order)
owner_acres.plot(kind='barh', ax=axes[1],
                 color=[owner_colors.get(o, '#95A5A6') for o in owner_order])
axes[1].set_xlabel('Total Acres')
axes[1].set_title('Burned Area by Land Owner')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))

# Right: Mean fire size
owner_mean = ca.groupby('Owner_Std')['FIRE_SIZE'].mean().sort_values(ascending=True)
owner_mean.plot(kind='barh', ax=axes[2],
                color=[owner_colors.get(o, '#95A5A6') for o in owner_mean.index])
axes[2].set_xlabel('Mean Fire Size (acres)')
axes[2].set_title('Mean Fire Size by Land Owner')

fig.suptitle('Figure 14: Land Ownership & Fire Patterns (FPA FOD unique)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/14_land_ownership')
plt.show()

In [ ]:
# Figure 15: County Geography (FPA FOD unique)
has_county = ca[ca['COUNTY'].notna()].copy()
print(f"Records with county info: {len(has_county):,} ({100*len(has_county)/len(ca):.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Left: Top 20 counties by fire count
county_counts = has_county.groupby('COUNTY').size().sort_values(ascending=False).head(20)
county_counts.plot(kind='barh', ax=axes[0], color='#3498DB', alpha=0.8)
axes[0].set_xlabel('Number of Fires')
axes[0].set_title('Top 20 Counties by Fire Count')
for i, val in enumerate(county_counts.values):
    axes[0].text(val + county_counts.max() * 0.01, i,
                 f'{val:,.0f}', va='center', fontsize=8)

# Right: Top 20 counties by total acres
county_acres = has_county.groupby('COUNTY')['FIRE_SIZE'].sum().sort_values(ascending=False).head(20)
county_acres.plot(kind='barh', ax=axes[1], color='#E63946', alpha=0.8)
axes[1].set_xlabel('Total Acres Burned')
axes[1].set_title('Top 20 Counties by Burned Area')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))

fig.suptitle('Figure 15: County Fire Geography (FPA FOD unique)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/15_county_geography')
plt.show()

In [ ]:
# Figure 16: Spatial Overview Map
import contextily as ctx

# Reproject to Albers for proper map
ca_albers = reproject_to_albers(ca)

fig, ax = plt.subplots(1, 1, figsize=(12, 14))

# Plot all fires, color by decade, size by log(acres)
decades = sorted(ca_albers['Decade'].dropna().unique())
decade_colors = plt.cm.viridis(np.linspace(0, 1, len(decades)))
decade_cmap = {d: c for d, c in zip(decades, decade_colors)}

for decade in decades:
    mask = ca_albers['Decade'] == decade
    subset = ca_albers[mask]
    # Subsample tiny fires for performance
    if len(subset) > 5000:
        subset = subset.sample(5000, random_state=42)
    sizes = np.clip(subset['Log10_FireSize'] * 2, 0.5, 50)
    ax.scatter(subset.geometry.x, subset.geometry.y,
               s=sizes, c=[decade_cmap[decade]], alpha=0.3,
               label=f"{int(decade)}s ({mask.sum():,})")

# Add basemap
try:
    ctx.add_basemap(ax, crs=ca_albers.crs.to_string(),
                    source=ctx.providers.CartoDB.Positron, zoom=6)
except Exception as e:
    print(f"Basemap failed: {e}")

# Label top 10 largest fires
top10 = ca_albers.nlargest(10, 'FIRE_SIZE')
for _, fire in top10.iterrows():
    name = fire.get('FIRE_NAME', 'Unknown')
    if pd.notna(name) and name != '':
        ax.annotate(f"{name}\n({fire['FIRE_YEAR']}, {format_acres(fire['FIRE_SIZE'])})",
                    xy=(fire.geometry.x, fire.geometry.y),
                    fontsize=7, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

ax.set_title('FPA FOD California — Spatial Overview (subsampled)', fontsize=13)
ax.legend(fontsize=8, loc='lower right', markerscale=3)
ax.set_axis_off()

plt.suptitle('Figure 16: Spatial Overview — All Fire Locations by Decade',
             fontsize=14, fontweight='bold', y=0.92)
plt.tight_layout()
save_figure(fig, 'fpa/16_spatial_overview')
plt.show()

### Interactive Map — Fires >= 100 Acres

In [ ]:
# Interactive Folium map (subsampled to fires >= 100 acres for performance)
import folium
from folium.plugins import MarkerCluster

# Filter to larger fires for interactive map
large_fires = ca[ca['FIRE_SIZE'] >= 100].copy()
large_fires_wgs = large_fires.to_crs(CRS_WGS84)
print(f"Fires >= 100 acres for interactive map: {len(large_fires_wgs):,}")

# Create map centered on California
m = folium.Map(location=[37.5, -119.5], zoom_start=6, tiles='CartoDB positron')
marker_cluster = MarkerCluster(name="Fires >= 100 ac").add_to(m)

cause_colors_folium = {'Human': 'red', 'Natural': 'green', 'Missing': 'gray'}

for _, fire in large_fires_wgs.iterrows():
    if pd.isna(fire.geometry.y) or pd.isna(fire.geometry.x):
        continue

    cause = fire.get('Cause_Label', 'Missing')
    color = cause_colors_folium.get(cause, 'gray')
    name = fire.get('FIRE_NAME', 'Unknown')
    if pd.isna(name):
        name = 'Unknown'

    popup_html = (
        f"<b>{name}</b><br>"
        f"Year: {fire['FIRE_YEAR']}<br>"
        f"Size: {fire['FIRE_SIZE']:,.1f} ac<br>"
        f"Cause: {fire.get('NWCG_GENERAL_CAUSE', 'N/A')}<br>"
        f"Owner: {fire.get('OWNER_DESCR', 'N/A')}<br>"
        f"County: {fire.get('COUNTY', 'N/A')}"
    )

    folium.CircleMarker(
        location=[fire.geometry.y, fire.geometry.x],
        radius=max(2, min(12, np.log10(fire['FIRE_SIZE'] + 1) * 3)),
        color=color, fill=True, fillColor=color, fillOpacity=0.5,
        popup=folium.Popup(popup_html, max_width=250),
    ).add_to(marker_cluster)

# Add legend
legend_html = '''
<div style="position:fixed; bottom:50px; left:50px; background:white;
     padding:10px; border:2px solid grey; z-index:1000; font-size:12px;">
<b>FPA FOD Fires >= 100 ac</b><br>
<span style="color:red;">●</span> Human<br>
<span style="color:green;">●</span> Natural<br>
<span style="color:gray;">●</span> Missing/Unknown
</div>'''
m.get_root().html.add_child(folium.Element(legend_html))

# Save
interactive_dir = OUTPUT_DIR / "interactive"
interactive_dir.mkdir(parents=True, exist_ok=True)
map_path = interactive_dir / "fpa_fires_interactive.html"
m.save(str(map_path))
print(f"Interactive map saved: {map_path}")
m

<a id='part9'></a>

---

# Part 9: Cross-Reference & ML Readiness

FPA FOD links to MTBS via the `MTBS_ID` field, enabling a **triple cross-reference**:
FPA FOD ↔ MTBS (direct linkage) ↔ CAL FIRE (via year/area correlation).

In [ ]:
# MTBS linkage analysis
mtbs_linked = ca[ca['has_mtbs_linkage']].copy()
print(f"FPA FOD records with MTBS linkage: {len(mtbs_linked):,}")
print(f"MTBS-linked total acres: {mtbs_linked['FIRE_SIZE'].sum():,.0f}")
print(f"Year range of linked records: {mtbs_linked['FIRE_YEAR'].min()} - {mtbs_linked['FIRE_YEAR'].max()}")
print(f"Mean size of linked fires: {mtbs_linked['FIRE_SIZE'].mean():,.0f} acres")
print(f"Min size of linked fires: {mtbs_linked['FIRE_SIZE'].min():,.0f} acres")

In [ ]:
# Figure 17: Triple Cross-Reference
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Left: Annual burned area comparison (FPA FOD vs MTBS-linked subset)
yearly_fpa = ca.groupby('FIRE_YEAR')['FIRE_SIZE'].sum()
yearly_mtbs = mtbs_linked.groupby('FIRE_YEAR')['FIRE_SIZE'].sum()

axes[0].plot(yearly_fpa.index, yearly_fpa.values, 'o-', color='#3498DB',
             label='FPA FOD (all CA fires)', linewidth=2, markersize=4)
axes[0].plot(yearly_mtbs.index, yearly_mtbs.values, 's-', color='#E63946',
             label='MTBS-linked subset', linewidth=2, markersize=4)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Acres Burned')
axes[0].set_title('Annual Burned Area: FPA FOD vs MTBS Subset')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))
axes[0].legend(fontsize=9)

# Correlation
common_years = yearly_fpa.index.intersection(yearly_mtbs.index)
r, p = stats.pearsonr(yearly_fpa[common_years], yearly_mtbs[common_years])
axes[0].annotate(f'r = {r:.3f}, p < 0.001',
                 xy=(0.05, 0.92), xycoords='axes fraction',
                 fontsize=10, fontweight='bold',
                 bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Middle: Scatter of FPA vs MTBS annual acres
axes[1].scatter(yearly_fpa[common_years], yearly_mtbs[common_years],
                c='#457B9D', s=60, alpha=0.7, edgecolors='white')
max_val = max(yearly_fpa[common_years].max(), yearly_mtbs[common_years].max())
axes[1].plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='1:1 line')
axes[1].set_xlabel('FPA FOD Annual Acres')
axes[1].set_ylabel('MTBS-Linked Annual Acres')
axes[1].set_title('Correlation: FPA FOD vs MTBS-Linked Acres')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: format_acres(x)))

# Add regression line
slope, intercept, r2, p2, se = stats.linregress(yearly_fpa[common_years], yearly_mtbs[common_years])
x_fit = np.linspace(0, max_val, 100)
axes[1].plot(x_fit, slope * x_fit + intercept, 'r-', alpha=0.5,
             label=f'R² = {r2**2:.3f}')
axes[1].legend()

# Right: Fire count comparison — FPA vs MTBS coverage
yearly_fpa_count = ca.groupby('FIRE_YEAR').size()
yearly_mtbs_count = mtbs_linked.groupby('FIRE_YEAR').size()

width = 0.35
x = np.arange(len(common_years))
axes[2].bar(x - width/2,
            yearly_fpa_count.reindex(common_years, fill_value=0).values,
            width, label='FPA FOD (all)', color='#3498DB', alpha=0.7)
axes[2].bar(x + width/2,
            yearly_mtbs_count.reindex(common_years, fill_value=0).values,
            width, label='MTBS-linked', color='#E63946', alpha=0.7)
axes[2].set_xticks(x[::5])
axes[2].set_xticklabels(common_years[::5])
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Number of Fires')
axes[2].set_title('Fire Count: FPA FOD vs MTBS-Linked')
axes[2].legend()
axes[2].set_yscale('log')

fig.suptitle('Figure 17: Triple Cross-Reference — FPA FOD ↔ MTBS',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fpa/17_cross_reference')
plt.show()

---

## ML Feature Engineering Opportunities from FPA FOD

FPA FOD adds several **new ML features** not available from CAL FIRE or MTBS:

### New Features for Phase 2–4
| Feature | Source Field | ML Value |
|---------|-------------|----------|
| **Fire count density** | Point locations | Historical ignition frequency per grid cell |
| **Cause probability** | NWCG_GENERAL_CAUSE | P(human) vs P(lightning) per cell |
| **Time-of-day risk** | DISCOVERY_TIME | Diurnal risk profile |
| **Duration prediction** | Duration_Days | Expected containment time |
| **Land ownership** | OWNER_DESCR | Management regime features |
| **Small fire density** | Size Class A-C | Frequency of minor ignitions |
| **MTBS severity linkage** | MTBS_ID | Connect point data to severity rasters |
| **County-level stats** | COUNTY/FIPS | Administrative unit features |

### Three-Dataset Integration Strategy
1. **Grid cell assignment**: Assign FPA FOD points to 800m grid cells
2. **Temporal aggregation**: Monthly fire count, mean size, dominant cause per cell
3. **MTBS enrichment**: For MTBS-linked fires, add severity from MTBS rasters
4. **CAL FIRE perimeters**: Add burned-area polygons for spatial extent
5. **Combined feature vector**: Each grid cell gets features from all three datasets

### Key Statistics Summary
| Metric | Value |
|--------|-------|
| California records | 251,881 |
| Year range | 1992–2020 |
| Total acres burned | ~20.9M |
| Fires < 10 acres | ~93% |
| Human-caused | ~77% |
| Lightning-caused | ~16% |
| MTBS-linked records | 1,447 |
| Counties covered | 58 |
| Land ownership categories | 17 |

---

*Analysis complete. Notebook generated as part of Phase 1.875 of the California Wildfire Prediction Project.*
*FPA FOD 6th Edition by Karen C. Short, USDA Forest Service (2022).*